In [1]:
import numpy as np
import pandas as pd

def calculate_bond_metrics(face_value, coupon_rate, market_rate, years, freq=12):
    """
    Calcule le Prix, la Duration de Macaulay et la Duration Modifiée.
    freq=12 pour des remboursements mensuels (standard en banque de détail).
    """
    periods = int(years * freq)
    per_period_market_rate = market_rate / freq
    per_period_coupon = (face_value * coupon_rate) / freq
    
    times = np.arange(1, periods + 1) / freq
    cash_flows = np.array([per_period_coupon] * (periods - 1) + [per_period_coupon + face_value])
    
    # Actualisation des flux (Discounting)
    discount_factors = (1 + per_period_market_rate)**-(times * freq)
    pv_cash_flows = cash_flows * discount_factors
    
    price = np.sum(pv_cash_flows)
    
    # Duration de Macaulay : Moyenne pondérée du temps
    macaulay_duration = np.sum(times * pv_cash_flows) / price
    
    # Duration Modifiée : Sensibilité au taux
    modified_duration = macaulay_duration / (1 + (market_rate / freq))
    
    return price, macaulay_duration, modified_duration

# --- SCÉNARIO BANQUE DES CARAÏBES ---
print("--- Simulation de Portefeuille ALM ---")

# La banque a prêté 100M€ à 5% sur 7 ans (taux fixe)
montant_total = 100_000_000
taux_fixe_client = 0.05
taux_marche_actuel = 0.04
maturite = 7

prix, mac_dur, mod_dur = calculate_bond_metrics(
    montant_total, taux_fixe_client, taux_marche_actuel, maturite
)

print(f"Prix actuel du portefeuille : {prix:,.2f} €")
print(f"Duration de Macaulay : {mac_dur:.2f} ans")
print(f"Duration Modifiée (Sensibilité) : {mod_dur:.2f}")

# --- STRESS TEST ---
# On simule une hausse des taux de la BCE de 100 bps (1%)
hausse_taux = 0.01
perte_estimee = -mod_dur * hausse_taux * prix

print(f"\n--- Stress Test : Hausse de {hausse_taux*100:.0f} bps ---")
print(f"Perte de valeur estimée de l'Actif : {perte_estimee:,.2f} €")

# --- STRATÉGIE DE COUVERTURE (SWAP) ---
# Pour s'immuniser, la banque doit prendre un Swap dont le gain 
# compensera cette perte de 6M€ si les taux montent.
print(f"\nConseil Stratégique :")
print(f"Pour protéger la banque, il faut mettre en place un Swap de taux")
print(f"avec un notionnel de {montant_total:,.0f} € et une duration cible de {mac_dur:.2f} ans.")

--- Simulation de Portefeuille ALM ---
Prix actuel du portefeuille : 106,096,606.48 €
Duration de Macaulay : 5.96 ans
Duration Modifiée (Sensibilité) : 5.94

--- Stress Test : Hausse de 1 bps ---
Perte de valeur estimée de l'Actif : -6,301,916.70 €

Conseil Stratégique :
Pour protéger la banque, il faut mettre en place un Swap de taux
avec un notionnel de 100,000,000 € et une duration cible de 5.96 ans.
